# Итоговое решение

Итоговая модель: среднее вероятностей трёх CatBoost с seed 0, 17 и 42 на 62 признаках. На проверочных разбиениях precision при recall не ниже 0.70 составила 0.7960 и 0.7131

In [1]:
import hashlib

import pandas as pd

import scripts

meta = scripts.read_meta(open_holdout=True)
clean = scripts.clean_events(scripts.window_events(scripts.read_events(), meta))
test = pd.read_csv(scripts.DATA_DIR / 'test.csv')
meta.shape, clean.shape, test.shape

((16000, 6), (283833, 20), (4909, 4))

## Признаки

Признаки посчитал отдельно для каждой cookie по очищенным событиям её окна. Для обучения использовал весь размеченный train, 04-06..04-19

In [2]:
features = scripts.model_features(clean, meta).join(
    scripts.indexed_by_cookie(meta)[['target', 'part']])
train = features[features['part'] != 'test']
predict = features.loc[pd.Index(test['cookie_id'], name='cookie_id')]
assert train['target'].notna().all() and predict['target'].isna().all()
print('обучение:', len(train), 'cookie,', int(train['target'].sum()), 'ботов |',
      'предсказание:', len(predict), 'cookie |', len(scripts.MODEL_FEATURES), 'признаков')

обучение: 11091 cookie, 899 ботов | предсказание: 4909 cookie | 62 признаков


## Обучение и предсказание

Три модели обучил на полном train с теми же параметрами: глубина 6, 1500 итераций, learning rate 0.03, l2 10, без ранней остановки. Для test усреднил их вероятности

In [3]:
model = scripts.final_model()
print('усреднение', len(model.estimators), 'CatBoost:', scripts.FINAL_PARAMS,
      '| seed', scripts.FINAL_SEEDS)
model.fit(train[scripts.MODEL_FEATURES], train['target'].astype(int))
score = model.predict_proba(predict[scripts.MODEL_FEATURES])[:, 1]
submission = pd.DataFrame({'cookie_id': test['cookie_id'], 'score': score})
submission.head(3)

усреднение 3 CatBoost: {'iterations': 1500, 'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 10} | seed (0, 17, 42)


,cookie_id,score
0,ck_315fb710a0e371e7,0.00052
1,ck_a76ee3b3e3e522fd,0.04646
2,ck_94c9a4d382689e82,0.02054


## Проверки и запись

В `submission.csv` две колонки, `cookie_id` и `score`, строки в порядке `test.csv`

In [4]:
assert list(submission.columns) == ['cookie_id', 'score']
assert submission['cookie_id'].tolist() == test['cookie_id'].tolist()
assert submission['cookie_id'].is_unique
assert submission['score'].notna().all()
assert submission['score'].between(0, 1).all()
submission.to_csv(scripts.REPO_ROOT / 'submission.csv', index=False)
digest = hashlib.sha256((scripts.REPO_ROOT / 'submission.csv').read_bytes()).hexdigest()
print('строк:', len(submission), '| SHA-256:', digest)

строк: 4909 | SHA-256: 0e324c5b74349993aad18cd90a4d7254217a5309ce7f6d9238f1d6b9ff6b3fb6
